# Silver — ticker

`bronze.trusts` + `bronze.trust_prices_yf` + `bronze.yf_pull_log` +
`silver.monthly_performance` → **`silver.ticker`**.

One row per instrument the project reports on: **the 99 UK trusts that are still listed, plus
SPY, IVV and VOO**. A trust the source has stopped pricing has been wound up, and it does not
appear here — but it is still recorded in `landing.yf_pull_log_raw`, with the source's own
refusal message against it, so the pipeline does not quietly agree that it never existed.

`status` is derived from coverage, never asserted: a trust is listed if the source still
priced it in the final month. The last priced month is read from **Bronze**, because
`monthly_performance` no longer holds the trusts that stopped and so cannot be asked who
stopped.

Flat, no SCD2: version history is built in Gold by MERGE. Run this **after**
`silver_etl_monthly_performance`, whose output supplies the coverage facts.

Spec: `specs/02_silver/silver.md`, amended by `specs/08_listed_only/listed-only.md`.

Expected: **102 rows**.

In [ ]:
CREATE OR REPLACE TEMP VIEW silver_stage_ticker AS
WITH latest AS (
  SELECT MAX(month_key) AS latest_month
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
),
coverage AS (
  SELECT ticker,
         MIN(month_key)    AS first_month,
         MAX(month_key)    AS last_month,
         COUNT(*)          AS months_available,
         MAX(price_source) AS price_source,
         MAX(currency)     AS currency
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
  GROUP BY ticker
),
source_span AS (
  -- The last month the source priced each trust. Read from Bronze, not from
  -- monthly_performance, because that table no longer holds the trusts that stopped —
  -- so it can no longer be asked who stopped. Delisting is a fact about coverage.
  SELECT REPLACE(symbol, '.L', '') AS ticker,
         CAST(DATE_FORMAT(MAX(TRUNC(TO_DATE(SUBSTRING(`Date`, 1, 10)), 'MM')),
                          'yyyyMM') AS INT) AS source_last_month
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
  GROUP BY 1
),
pull AS (
  SELECT source_ticker, MAX(status) AS pull_status,
         MAX(CAST(row_count AS INT)) AS pull_rows
  FROM `index-vs-trust-pipeline`.bronze.yf_pull_log
  GROUP BY source_ticker
),
meta AS (
  -- Two rows carry a blank ticker (Island Innovation, Witan) and cannot be keyed.
  SELECT ticker, trust_name, aic_sector, manager, management_group, source_url
  FROM `index-vs-trust-pipeline`.bronze.trusts
  WHERE ticker IS NOT NULL AND TRIM(ticker) <> ''
),
trusts AS (
  SELECT m.ticker,
         m.trust_name,
         'Trust'                          AS entity_type,
         m.aic_sector,
         m.manager,
         m.management_group,
         m.source_url,
         -- A scalar expression, not a window function. Buys a free insight: do
         -- solo-managed trusts beat the index more often than committees do?
         CASE WHEN m.manager IS NULL OR TRIM(m.manager) = '' THEN NULL
              WHEN SIZE(SPLIT(m.manager, ', ')) > 1          THEN 'multi'
              ELSE 'sole' END             AS manager_structure,
         c.currency,
         COALESCE(c.price_source, 'none') AS price_source,
         c.first_month,
         c.last_month,
         COALESCE(c.months_available, 0)  AS months_available
  FROM meta m
  LEFT JOIN coverage c ON c.ticker = m.ticker
),
index_rows AS (
  SELECT c.ticker,
         c.ticker             AS trust_name,
         'Index'              AS entity_type,
         CAST(NULL AS STRING) AS aic_sector,
         CAST(NULL AS STRING) AS manager,
         CAST(NULL AS STRING) AS management_group,
         CAST(NULL AS STRING) AS source_url,
         CAST(NULL AS STRING) AS manager_structure,
         c.currency, c.price_source, c.first_month, c.last_month, c.months_available
  FROM coverage c
  WHERE c.ticker IN ('SPY', 'IVV', 'VOO')
),
combined AS (
  SELECT * FROM trusts
  UNION ALL
  SELECT * FROM index_rows
),
scored AS (
  SELECT combined.*,
         -- Derived, never asserted. The seed file claimed an is_active column that
         -- contradicted the prices, which is why that file was dropped.
         CASE WHEN pull.pull_status = 'NODATA'                     THEN 'delisted'
              WHEN src.source_last_month < latest.latest_month     THEN 'delisted'
              ELSE 'active' END AS status,
         -- The 36-month floor is a Gold rule; Silver only labels who would fail it.
         -- excluded: Yahoo returned a full history that Silver could not trust (S2d).
         CASE WHEN combined.months_available >= 36 THEN 'usable'
              WHEN combined.months_available > 0  THEN 'stub'
              WHEN pull.pull_rows > 36            THEN 'excluded'
              ELSE 'no-data' END AS data_status
  FROM combined
  CROSS JOIN latest
  LEFT JOIN pull ON pull.source_ticker = combined.ticker
  LEFT JOIN source_span src ON src.ticker = combined.ticker
)
-- Listed only. The dimension is the universe the project reports on, and a trust that has
-- been wound up is not in it.
SELECT * FROM scored WHERE status = 'active';

In [ ]:
MERGE INTO `index-vs-trust-pipeline`.silver.ticker AS t
USING silver_stage_ticker AS s
   ON t.ticker = s.ticker
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
-- A hard delete, on purpose. The table is defined as the trusts that are still listed, so
-- one that winds up has to leave it — a filter alone would only stop new rows arriving.
WHEN NOT MATCHED BY SOURCE THEN DELETE;

## Verification

In [ ]:
SELECT COUNT(*)                                                    AS rows_total,
       SUM(CASE WHEN entity_type = 'Trust'    THEN 1 ELSE 0 END)   AS trusts,
       SUM(CASE WHEN data_status = 'usable'   THEN 1 ELSE 0 END)   AS usable,
       SUM(CASE WHEN data_status = 'stub'     THEN 1 ELSE 0 END)   AS stub,
       SUM(CASE WHEN data_status = 'no-data'  THEN 1 ELSE 0 END)   AS no_data,
       SUM(CASE WHEN data_status = 'excluded' THEN 1 ELSE 0 END)   AS excluded,
       SUM(CASE WHEN price_source = 'none'    THEN 1 ELSE 0 END)   AS unpriced,
       SUM(CASE WHEN status <> 'active'       THEN 1 ELSE 0 END)   AS not_listed
FROM `index-vs-trust-pipeline`.silver.ticker;

Expect **102 / 99 / 92 / 1 / 2 / 7 / 9 / 0**.

- **`not_listed` must be 0.** That is the whole point of this step: the dimension holds the
  trusts that are still listed, and nothing else.
- **usable 92 includes the 3 index tickers, so 89 trusts are usable.**
- **excluded 7** — `CLDN`, `JEMA`, `MRC`, `MYI`, `NAS`, `PCT`, `WWH`. Yahoo offers a full
  history for each, so they are not "no data"; the history simply cannot be trusted. They are
  still listed, so they stay in the dimension and are named openly.
- **stub 1** — `BSIF`, one month, below the 36-month floor Gold applies.
- **no-data 2** — `MNTN` (its only bar is in the excluded partial month) and `EOT` (its one
  dividend was 155% of its price, so its rows went). Both are still listed.
- **unpriced 9** — the 7 excluded plus those 2. Every one is a trust the dimension names and
  the results cannot include, which is what `semantic.v_universe` puts on screen.

**19 rows left**: the 18 Yahoo returns nothing for, plus `ADIG`, whose single bar stops in
2026-03. They are still visible in `landing.yf_pull_log_raw`, with the source's own refusal
message against each — the pipeline still records that they existed, it just does not report
on them.

In [0]:
-- Nulls are carried, never defaulted. Substituting 'Unknown' would invent a management
-- group that dim_ticker would then open an SCD2 version on.
SELECT SUM(CASE WHEN manager_structure = 'multi' THEN 1 ELSE 0 END) AS multi_manager,
       SUM(CASE WHEN manager_structure = 'sole'  THEN 1 ELSE 0 END) AS sole_manager,
       SUM(CASE WHEN manager_structure IS NULL
                 AND entity_type = 'Trust'       THEN 1 ELSE 0 END) AS no_manager,
       SUM(CASE WHEN data_status <> 'no-data'
                 AND manager IS NULL             THEN 1 ELSE 0 END) AS priced_but_no_manager,
       COUNT(DISTINCT management_group)                             AS management_groups
FROM `index-vs-trust-pipeline`.silver.ticker;

Expect **70 / 27 / 2 / 3 / 52**.

- 70 + 27 = **97** of the 99 listed trusts have a named manager. The other **2** carry a blank
  manager in the metadata file, so `manager_structure` is null — carried as a null, never
  defaulted to "Unknown".
- **`priced_but_no_manager` is 3, and those 3 are `SPY`, `IVV` and `VOO`.** An index has no
  manager by definition, so this number should be exactly 3. A fourth would mean a trust we
  price had lost its metadata, which is what the check is for.
- `no_manager` fell from 21 to 2 with the delisted trusts: all 19 were wound up, so the
  metadata file carries no manager for any of them. Multi and sole are unchanged, because
  none of the 19 had a manager to count.

In [ ]:
-- The trusts the metadata names but the results cannot include, and why each one is out.
-- Named openly rather than quietly dropped: every one is still listed, so it belongs in
-- the dimension even though no horizon can be measured for it.
SELECT ticker, trust_name, management_group, data_status, price_source, months_available
FROM `index-vs-trust-pipeline`.silver.ticker
WHERE entity_type = 'Trust'
  AND data_status <> 'usable'
ORDER BY data_status, ticker;

Expect **10 rows** — the 7 `excluded`, the 2 `no-data` and the 1 `stub`:

| data_status | trusts |
|---|---|
| `excluded` | `CLDN`, `JEMA`, `MRC`, `MYI`, `NAS`, `PCT`, `WWH` |
| `no-data` | `EOT`, `MNTN` |
| `stub` | `BSIF` |

89 usable trusts plus these 10 is the 99 listed trusts in the dimension.

This is the table to put on screen when asked *"which trusts are missing from your results,
and why?"* — every one of them is named, with the reason attached, rather than being silently
absent from a count.